# Optimization Lab: Unconstrained + Lagrange Multipliers (Lecture 5)

Notebook นี้ให้คุณจำลองและเปรียบเทียบเนื้อหาจากเลคเชอร์ 5 ได้เอง:

1. **Single-variable optimization** — หา critical point ด้วย f'(x)=0 แล้วเช็ค f''(x)
2. **Multi-variable + Hessian test** — ฟังก์ชันทั่วไป: หา ∇f=0 แล้วจัด max/min/saddle อัตโนมัติ
3. **ตรวจคำตอบด้วย scipy.optimize** (numerical) เทียบกับ sympy (symbolic)
4. **Lagrange Multipliers** — ฟังก์ชันทั่วไปสำหรับ constrained optimization พร้อมตัวอย่างจากสไลด์ทั้ง 4 ข้อ
5. ช่องท้ายไฟล์ให้ลองใส่ฟังก์ชันของคุณเอง


In [ ]:
import sympy as sp
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sp.init_printing()


## 1. Single-variable optimization

ตัวอย่างจากสไลด์: l(x) = x⁴ + 7x³ + 5x² − 17x + 3


In [ ]:
x = sp.symbols('x')
l = x**4 + 7*x**3 + 5*x**2 - 17*x + 3

dl = sp.diff(l, x)
d2l = sp.diff(l, x, 2)
print("l'(x)  =", dl)
print("l''(x) =", d2l)

critical_points_raw = sp.solve(sp.Eq(dl, 0), x)
# แปลงเป็นตัวเลขและกรองเฉพาะรากจริง (สมการดีกรี 3 ให้ค่าในรูป symbolic ที่อ่านยาก จึงประเมินเป็นตัวเลข)
critical_points = []
for cp in critical_points_raw:
    val = complex(cp.evalf())
    if abs(val.imag) < 1e-9:
        critical_points.append(sp.Float(val.real))
print("Critical points (real, ตัวเลข):", [round(float(c), 4) for c in critical_points])

for cp in critical_points:
    second = d2l.subs(x, cp)
    kind = "local minimum" if second > 0 else ("local maximum" if second < 0 else "inconclusive (ต้องเช็คเพิ่ม)")
    print(f"x = {float(cp):.4f} -> l''(x) = {float(second):.4f} -> {kind}")


In [ ]:
# วาดกราฟพร้อม critical points
f_num = sp.lambdify(x, l, 'numpy')
xs = np.linspace(-6, 2, 400)
ys = f_num(xs)

plt.figure(figsize=(6, 4))
plt.plot(xs, ys)
for cp in critical_points:
    if cp.is_real:
        cpf = float(cp)
        plt.plot(cpf, f_num(cpf), 'ro')
        plt.annotate(f"({cpf:.2f}, {f_num(cpf):.1f})", (cpf, f_num(cpf)), textcoords="offset points", xytext=(6, 6))
plt.xlabel("x")
plt.ylabel("l(x)")
plt.title("l(x) = x^4 + 7x^3 + 5x^2 - 17x + 3")
plt.grid(alpha=0.3)
plt.show()

# เทียบกับ scipy (numerical, ต้องลองหลาย initial point เพราะ non-convex)
for x0 in [-6, -2, 0, 2]:
    res = minimize(lambda v: f_num(v[0]), x0=[x0])
    print(f"start x0={x0:>3} -> scipy พบ x* = {res.x[0]:.4f}, f(x*) = {res.fun:.4f}")


## 2. Multi-variable: Hessian test (generic function)

ฟังก์ชันนี้ใช้ได้กับ f(x,y) ใดๆ: หา ∇f=0 ด้วย sympy, คำนวณ Hessian, แล้วจัดประเภทแต่ละ critical point อัตโนมัติ


In [ ]:
xs_, ys_ = sp.symbols('x y')

def analyze_critical_points(f_expr, variables=(xs_, ys_), verbose=True):
    """
    f_expr : sympy expression ของ f(x, y)
    คืนค่า list ของ dict {point, fxx, fyy, fxy, D, classification}
    """
    grad = [sp.diff(f_expr, v) for v in variables]
    solutions = sp.solve(grad, variables, dict=True)

    fxx = sp.diff(f_expr, variables[0], 2)
    fyy = sp.diff(f_expr, variables[1], 2)
    fxy = sp.diff(f_expr, variables[0], variables[1])

    results = []
    for sol in solutions:
        pt = tuple(sol[v] for v in variables)
        if not all(v.is_real for v in pt):
            continue
        Hxx = fxx.subs(sol)
        Hyy = fyy.subs(sol)
        Hxy = fxy.subs(sol)
        D = Hxx * Hyy - Hxy**2

        if D < 0:
            kind = "Saddle point"
        elif D > 0 and Hxx > 0:
            kind = "Local minimum"
        elif D > 0 and Hxx < 0:
            kind = "Local maximum"
        else:
            kind = "Inconclusive (D=0)"

        results.append({"point": pt, "fxx": Hxx, "fyy": Hyy, "fxy": Hxy, "D": D, "kind": kind})
        if verbose:
            print(f"Critical point {pt}: f_xx={Hxx}, f_yy={Hyy}, f_xy={Hxy}, D={D}  ->  {kind}")

    return results


In [ ]:
# ตัวอย่างสไลด์: f(x,y) = x^3 - y^3 + 9xy
f1 = xs_**3 - ys_**3 + 9*xs_*ys_
print("f(x,y) =", f1, "\n")
results1 = analyze_critical_points(f1)


In [ ]:
# ตัวอย่างสไลด์: f(x,y) = 3y^2 - 2y^3 - 3x^2 + 6xy
f2 = 3*ys_**2 - 2*ys_**3 - 3*xs_**2 + 6*xs_*ys_
print("f(x,y) =", f2, "\n")
results2 = analyze_critical_points(f2)


In [ ]:
# ตัวอย่างสไลด์: f(x,y) = x^2 + y^2 - 4x - 6y + 20  (ไม่มี saddle เพราะเป็น convex bowl)
f3 = xs_**2 + ys_**2 - 4*xs_ - 6*ys_ + 20
print("f(x,y) =", f3, "\n")
results3 = analyze_critical_points(f3)

# เทียบกับ scipy
f3_num = sp.lambdify((xs_, ys_), f3, 'numpy')
res = minimize(lambda v: f3_num(v[0], v[1]), x0=[0, 0])
print("\nscipy พบจุดต่ำสุดที่:", res.x, "ค่า f =", res.fun)


### Plot แบบหมุนได้ (interactive)

`matplotlib` แบบ static หมุนไม่ได้ในบางสภาพแวดล้อม (เช่น VS Code / Colab บาง cell) จึงเปลี่ยนมาใช้
`plotly` ซึ่งให้กราฟที่ **ลากเมาส์หมุนได้จริง** ในเอาต์พุตของ notebook โดยตรง (ไม่ต้องติดตั้งอะไรเพิ่มถ้ามี
`plotly` อยู่แล้ว — ถ้ายังไม่มีให้รัน `pip install plotly` ก่อน)


In [ ]:
import plotly.graph_objects as go

def plot_surface_3d(f_expr, results, xr=(-5, 5), yr=(-5, 5), title=""):
    f_num = sp.lambdify((xs_, ys_), f_expr, 'numpy')
    X, Y = np.meshgrid(np.linspace(*xr, 60), np.linspace(*yr, 60))
    Z = f_num(X, Y)

    fig = go.Figure()

    # พื้นผิวหลัก
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z,
        colorscale="Viridis",
        opacity=0.9,
        showscale=False,
        contours={"z": {"show": True, "usecolormap": True, "project": {"z": True}}},
    ))

    # จุด critical points แต่ละประเภท ใช้สีต่างกัน
    colors = {"Local minimum": "blue", "Local maximum": "red", "Saddle point": "black"}
    seen_kinds = set()
    for r in results:
        px, py = float(r["point"][0]), float(r["point"][1])
        pz = float(f_num(px, py))
        kind = r["kind"]
        fig.add_trace(go.Scatter3d(
            x=[px], y=[py], z=[pz],
            mode="markers",
            marker={"size": 6, "color": colors.get(kind, "gray")},
            name=kind,
            showlegend=(kind not in seen_kinds),
        ))
        seen_kinds.add(kind)

    fig.update_layout(
        title=title,
        scene={"xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "f(x,y)"},
        margin={"l": 0, "r": 0, "t": 40, "b": 0},
        legend={"x": 0.02, "y": 0.98},
        width=650, height=550,
    )
    fig.show()
    return fig

plot_surface_3d(f1, results1, xr=(-6, 6), yr=(-6, 6), title="f(x,y) = x^3 - y^3 + 9xy")
plot_surface_3d(f2, results2, xr=(-4, 5), yr=(-4, 5), title="f(x,y) = 3y^2 - 2y^3 - 3x^2 + 6xy")


## 3. Lagrange Multipliers (generic function)

แก้ปัญหา: optimize f(x,y) subject to g(x,y) = 0 โดยตั้งสมการ ∇f = λ∇g และ g = 0 แล้วให้ sympy แก้ระบบสมการ


In [ ]:
lam = sp.symbols('lambda')

def lagrange_solve(f_expr, g_expr, variables=(xs_, ys_), verbose=True):
    """
    f_expr : objective function f(x, y)
    g_expr : constraint ในรูป g(x, y) = 0
    คืนค่า list ของ dict {point, f_value}
    """
    grad_f = [sp.diff(f_expr, v) for v in variables]
    grad_g = [sp.diff(g_expr, v) for v in variables]

    equations = [sp.Eq(grad_f[i], lam * grad_g[i]) for i in range(len(variables))]
    equations.append(sp.Eq(g_expr, 0))

    solutions = sp.solve(equations, list(variables) + [lam], dict=True)

    results = []
    for sol in solutions:
        pt = tuple(sol[v] for v in variables)
        if not all(v.is_real for v in pt):
            continue
        f_val = f_expr.subs(sol)
        results.append({"point": pt, "f_value": f_val})
        if verbose:
            print(f"(x, y) = ({pt[0]}, {pt[1]})  ->  f = {f_val}   (lambda = {sol[lam]})")

    return results


In [ ]:
# Example 1: f(x,y) = xy, g: x^2/8 + y^2/2 - 1 = 0
f_e1 = xs_ * ys_
g_e1 = xs_**2 / 8 + ys_**2 / 2 - 1
print("=== Example 1: f=xy บนวงรี x^2/8 + y^2/2 = 1 ===")
res_e1 = lagrange_solve(f_e1, g_e1)


In [ ]:
# Example 2: f(x,y) = 3x + 4y, g: x^2 + y^2 - 1 = 0
f_e2 = 3*xs_ + 4*ys_
g_e2 = xs_**2 + ys_**2 - 1
print("=== Example 2: f=3x+4y บนวงกลม x^2+y^2=1 ===")
res_e2 = lagrange_solve(f_e2, g_e2)


In [ ]:
# Example 3: f(x,y) = x^2 + 4y^2 - 2x + 8y, g: x + 2y - 7 = 0
f_e3 = xs_**2 + 4*ys_**2 - 2*xs_ + 8*ys_
g_e3 = xs_ + 2*ys_ - 7
print("=== Example 3: f=x^2+4y^2-2x+8y, constraint x+2y=7 ===")
res_e3 = lagrange_solve(f_e3, g_e3)


In [ ]:
# Example 4 (golf ball profit): f(x,y) = 48x + 96y - x^2 - 2xy - 9y^2, g: 20x + 4y - 216 = 0
f_e4 = 48*xs_ + 96*ys_ - xs_**2 - 2*xs_*ys_ - 9*ys_**2
g_e4 = 20*xs_ + 4*ys_ - 216
print("=== Example 4: Pro-T golf ball profit ===")
res_e4 = lagrange_solve(f_e4, g_e4)


In [ ]:
# เทียบกับ scipy.optimize.minimize (SLSQP รองรับ equality constraint) สำหรับ Example 4
f_e4_num = sp.lambdify((xs_, ys_), -f_e4, 'numpy')  # ใส่ลบเพราะ scipy minimize
constraint = {'type': 'eq', 'fun': lambda v: 20*v[0] + 4*v[1] - 216}

res = minimize(lambda v: f_e4_num(v[0], v[1]), x0=[5, 5], constraints=[constraint])
print("scipy (SLSQP):", res.x, "-> profit =", -res.fun)
print("sympy (Lagrange):", res_e4)


## 4. ลองฟังก์ชันของคุณเอง

แก้ `f_custom` และ `g_custom` ด้านล่างแล้วรันใหม่ — ใช้ได้ทั้ง unconstrained (Hessian test) และ constrained (Lagrange)


In [ ]:
# ==== Unconstrained: แก้ f_custom แล้วรัน ====
f_custom = xs_**2 * ys_ - 3*xs_*ys_**2 + ys_**3

print("Unconstrained analysis:")
results_custom = analyze_critical_points(f_custom)


In [ ]:
# ==== Constrained (Lagrange): แก้ f_custom2 / g_custom แล้วรัน ====
f_custom2 = xs_**2 + ys_**2
g_custom = xs_ + ys_ - 4

print("Constrained (Lagrange) analysis:")
res_custom = lagrange_solve(f_custom2, g_custom)
